In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.home() / 'Downloads' / 'idx exchange' / 'csv'
MONTHS = pd.period_range('2024-01', '2026-04', freq='M')

print('Data folder:', DATA_DIR)
print('Folder exists:', DATA_DIR.exists())
print('Number of months:', len(MONTHS))

Data folder: /Users/jasmineyuan/Downloads/idx exchange/csv
Folder exists: True
Number of months: 28


In [2]:
# Select monthly CRMLS Sold files, excluding filled versions.

from pathlib import Path
import pandas as pd

# Set the data folder.
DATA_DIR = Path.home() / "Downloads" / "idx exchange" / "csv"

print("Data folder:", DATA_DIR)
print("Folder exists:", DATA_DIR.exists())

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}")

# Select all CRMLS Sold CSV files.

selected_files = sorted(DATA_DIR.glob("CRMLSSold*.csv"))

# Create a list to store the monthly DataFrames.
frames = []

# Read each selected CSV file.
for path in selected_files:
    monthly_df = pd.read_csv(
        path,
        low_memory=False,
        dtype={"PostalCode": "string"},
    )

    # Add the source file month.
   
    monthly_df["source_month"] = (
    path.stem.removeprefix("CRMLSSold").removesuffix("_filled")
)

    # Store the monthly DataFrame.
    frames.append(monthly_df)

# Stop if no matching files were loaded.
if not frames:
    raise ValueError("No matching CSV files were found.")

# Combine all monthly DataFrames.
df = pd.concat(frames, ignore_index=True)

# Display the loading results.
print("Number of files loaded:", len(frames))
print("Combined dataset shape (rows, columns):", df.shape)

display(
    df.groupby("source_month")
    .size()
    .rename("Row count")
)

Data folder: /Users/jasmineyuan/Downloads/idx exchange/csv
Folder exists: True
Number of files loaded: 28
Combined dataset shape (rows, columns): (615707, 85)


source_month
202401    17958
202402    19925
202403    23276
202404    24640
202405    26487
202406    24328
202407    26240
202408    24558
202409    21267
202410    23274
202411    20279
202412    20241
202501    18738
202502    18702
202503    21445
202504    23262
202505    23154
202506    22883
202507    23646
202508    22972
202509    22443
202510    23233
202511    19088
202512    20538
202601    16487
202602    19010
202603    23372
202604    24261
Name: Row count, dtype: int64

In [3]:
# *only use information that’s available for any address at the time of prediction.
# * Property characteristics (square footage, beds, baths, lot size, age, location, school district, comp-based neighborhood prices) pass that test.


In [4]:
# explicity select the property feastures 

In [5]:

# Keep California single-family residential properties.
single_family_df = df.loc[
    df["PropertyType"].eq("Residential")
    & df["PropertySubType"].eq("SingleFamilyResidence")
    & df["StateOrProvince"].eq("CA")
].copy()

print("California single-family rows:", len(single_family_df))

California single-family rows: 309713


In [6]:
# Fields that depend on a listing or transaction.
excluded_columns = [
    # Listing prices and market duration
    "OriginalListPrice",
    "ListPrice",
    "DaysOnMarket",

    # Listing identifiers and status
    "ListingKey",
    "ListingKeyNumeric",
    "ListingId",
    "MlsStatus",

    # Listing and transaction dates
    "ContractStatusChangeDate",
    "PurchaseContractDate",
    "ListingContractDate",

    # Listing agents and offices
    "ListAgentEmail",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFullName",
    "ListOfficeName",
    "CoListOfficeName",
    "CoListAgentFirstName",
    "CoListAgentLastName",

    # Buyer agents and offices
    "BuyerOfficeName",
    "BuyerOfficeAOR",
    "BuyerAgentMlsId",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "CoBuyerAgentFirstName",

    # Transaction compensation
    "BuyerAgencyCompensationType",
    "BuyerAgencyCompensation",
    
    # Agent affiliations and MLS source metadata
    "BuyerAgentAOR",
    "ListAgentAOR",
    "OriginatingSystemName",
    "OriginatingSystemSubName",

]

# Keep the target and sale dates separately.
y = single_family_df["ClosePrice"].copy()
sale_dates = single_family_df["CloseDate"].copy()

# Keep source metadata outside the property audit.
metadata_columns = [
    "source_month",
    "source_file",
    "source_is_filled",
]

# Retain the remaining columns for missing-value inspection.
audit_columns = [
    col for col in single_family_df.columns
    if col not in excluded_columns
    + ["ClosePrice", "CloseDate"]
    + metadata_columns
]

property_data = single_family_df[audit_columns].copy()

# Convert blank strings to missing values.
property_data = property_data.replace(r"^\s*$", pd.NA, regex=True)

# Summarize missing values.
missing_summary = pd.DataFrame({
    "Data type": property_data.dtypes.astype(str),
    "Missing count": property_data.isna().sum(),
    "Missing percentage": (
        property_data.isna().mean() * 100
    ).round(2),
}).sort_values("Missing percentage", ascending=False)

print("Total single-family rows:", len(property_data))
print("Number of retained columns:", property_data.shape[1])

display(missing_summary)

# Inspect missing target values separately.
print("Missing ClosePrice count:", y.isna().sum())

Total single-family rows: 309713
Number of retained columns: 52


,Data type,Missing count,Missing percentage
TaxYear,float64,309713,100.00
TaxAnnualAmount,float64,309713,100.00
MiddleOrJuniorSchoolDistrict,float64,309713,100.00
BusinessType,str,309713,100.00
FireplacesTotal,float64,309713,100.00
ElementarySchoolDistrict,float64,309713,100.00
AboveGradeFinishedArea,float64,309713,100.00
CoveredSpaces,float64,309713,100.00
WaterfrontYN,object,309577,99.96
BelowGradeFinishedArea,float64,307811,99.39


Missing ClosePrice count: 2


In [7]:
# Identify columns with no observed values.
all_missing_columns = property_data.columns[
    property_data.isna().all()
].tolist()

print("Columns dropped because all values are missing:")
print(all_missing_columns)

# Drop empty columns while retaining all rows.
property_data = property_data.drop(columns=all_missing_columns)

print("Remaining columns:", property_data.shape[1])
print("Remaining rows:", len(property_data))

Columns dropped because all values are missing:
['FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'TaxYear', 'ElementarySchoolDistrict', 'BusinessType', 'CoveredSpaces', 'MiddleOrJuniorSchoolDistrict']
Remaining columns: 44
Remaining rows: 309713


In [8]:
# Inspect the values in the added columns.
for col in ["latfilled", "lonfilled"]:
    print(f"\nColumn: {col}")
    print(property_data[col].value_counts(dropna=False).head(10))


Column: latfilled
latfilled
NaN      229123
False     46782
True      33808
Name: count, dtype: int64

Column: lonfilled
lonfilled
NaN      229123
False     46782
True      33808
Name: count, dtype: int64


In [9]:
# Remove coordinate-fill flags.
property_data = property_data.drop(columns=["latfilled", "lonfilled"])

In [10]:
import numpy as np

# Convert the target to numeric values.
y = pd.to_numeric(y, errors="coerce")

# Identify valid close prices.
valid_target = y.notna() & np.isfinite(y) & y.gt(0)

print("Rows removed due to invalid ClosePrice:", (~valid_target).sum())

# Keep property data, target, and sale dates aligned.
property_data = property_data.loc[valid_target].copy()
y = y.loc[valid_target].copy()
sale_dates = sale_dates.loc[valid_target].copy()

print("Remaining rows:", len(property_data))
print("Missing ClosePrice count:", y.isna().sum())

Rows removed due to invalid ClosePrice: 3
Remaining rows: 309710
Missing ClosePrice count: 0


In [11]:
# Verify row counts and index alignment.
print("Row counts:", len(property_data), len(y), len(sale_dates))
print(
    "Indexes aligned:",
    property_data.index.equals(y.index)
    and property_data.index.equals(sale_dates.index)
)

Row counts: 309710 309710 309710
Indexes aligned: True


In [12]:
# Inspect columns with more than 90% missing values.
missing_rate = property_data.isna().mean()

high_missing_columns = missing_rate[
    missing_rate > 0.90
].index.tolist()

high_missing_summary = pd.DataFrame({
    "Missing percentage": (
        missing_rate[high_missing_columns] * 100
    ).round(2),
    "Non-missing count": (
        property_data[high_missing_columns].notna().sum()
    ),
}).sort_values("Missing percentage", ascending=False)

display(high_missing_summary)

# Inspect observed values before deciding how to handle each column.
for col in high_missing_columns:
    print(f"\nColumn: {col}")
    print(property_data[col].value_counts().head(10))

,Missing percentage,Non-missing count
WaterfrontYN,99.96,136
BelowGradeFinishedArea,99.39,1902
BasementYN,97.59,7454
BuilderName,95.33,14458
LotSizeDimensions,93.88,18960
BuildingAreaTotal,93.65,19655



Column: WaterfrontYN
WaterfrontYN
True    136
Name: count, dtype: int64

Column: BasementYN
BasementYN
True    7454
Name: count, dtype: int64

Column: BuilderName
BuilderName
Lennar               944
Richmond American    661
D.R. Horton          610
Meritage Homes       504
DR Horton            496
Toll Brothers        450
Taylor Morrison      430
Shea Homes           367
D. R. Horton         351
Pulte Homes          268
Name: count, dtype: int64

Column: BuildingAreaTotal
BuildingAreaTotal
0.0       94
1600.0    44
1200.0    40
1400.0    38
1800.0    38
1500.0    38
1680.0    38
2000.0    34
1260.0    32
4000.0    30
Name: count, dtype: int64

Column: BelowGradeFinishedArea
BelowGradeFinishedArea
0.0       1532
1.0        158
600.0       11
100.0       10
500.0        8
1000.0       6
200.0        5
800.0        5
400.0        4
300.0        4
Name: count, dtype: int64

Column: LotSizeDimensions
LotSizeDimensions
50x135    347
50x100    248
50x120    235
50x130    223
50x150    203
5

In [13]:
# keep to remove those properties

In [14]:
# Drop sparse columns from the initial feature dataset.
sparse_columns_to_drop = [
    "WaterfrontYN",
    "BasementYN",
    "BuilderName",
    "BuildingAreaTotal",
    "BelowGradeFinishedArea",
    "LotSizeDimensions",
]

property_data = property_data.drop(columns=sparse_columns_to_drop)

print("Remaining columns:", property_data.shape[1])
print("Remaining rows:", len(property_data))

Remaining columns: 36
Remaining rows: 309710


In [15]:
# Inspect the values in remaining Yes/No columns.
yn_columns = [
    col for col in property_data.columns
    if col.endswith("YN")
]

for col in yn_columns:
    print(f"\nColumn: {col}")
    print(property_data[col].value_counts(dropna=False))


Column: ViewYN
ViewYN
True     170254
False    111845
NaN       27611
Name: count, dtype: int64

Column: PoolPrivateYN
PoolPrivateYN
False    234250
True      45669
NaN       29791
Name: count, dtype: int64

Column: AttachedGarageYN
AttachedGarageYN
True     227955
False     45720
NaN       36035
Name: count, dtype: int64

Column: FireplaceYN
FireplaceYN
True     225055
False     84449
NaN         206
Name: count, dtype: int64

Column: NewConstructionYN
NewConstructionYN
False    276172
NaN       22487
True      11051
Name: count, dtype: int64


In [16]:
# Mark missing Yes/No values as "Unknown".
for col in yn_columns:
    property_data[col] = (
        property_data[col]
        .astype("string")
        .fillna("Unknown")
    )

# Verify the categories and counts.
for col in yn_columns:
    print(f"\nColumn: {col}")
    print(property_data[col].value_counts(dropna=False))

print("\nRemaining rows:", len(property_data))


Column: ViewYN
ViewYN
True       170254
False      111845
Unknown     27611
Name: count, dtype: Int64

Column: PoolPrivateYN
PoolPrivateYN
False      234250
True        45669
Unknown     29791
Name: count, dtype: Int64

Column: AttachedGarageYN
AttachedGarageYN
True       227955
False       45720
Unknown     36035
Name: count, dtype: Int64

Column: FireplaceYN
FireplaceYN
True       225055
False       84449
Unknown       206
Name: count, dtype: Int64

Column: NewConstructionYN
NewConstructionYN
False      276172
Unknown     22487
True        11051
Name: count, dtype: Int64

Remaining rows: 309710


In [17]:
# Find text columns other than the Yes/No fields.
text_columns = [
    col
    for col in property_data.select_dtypes(
        include=["object", "string"]
    ).columns
    if col not in yn_columns
]

# Inspect missing values in these columns.
text_missing_summary = pd.DataFrame({
    "Missing count": property_data[text_columns].isna().sum(),
    "Missing percentage": (
        property_data[text_columns].isna().mean() * 100
    ).round(2),
})

In [18]:
display(text_missing_summary)

,Missing count,Missing percentage
Flooring,110708,35.75
UnparsedAddress,294,0.09
PropertyType,0,0.00
AssociationFeeFrequency,231900,74.88
MLSAreaMajor,44008,14.21
CountyOrParish,0,0.00
ElementarySchool,267338,86.32
PropertySubType,0,0.00
SubdivisionName,202363,65.34
City,240,0.08


In [19]:
# Show remaining missing values across all retained columns.
remaining_missing = property_data.isna().sum()

display(
    remaining_missing[remaining_missing > 0]
    .sort_values(ascending=False)
    .rename("Missing count")
)

ElementarySchool           267338
MiddleOrJuniorSchool       267009
HighSchool                 254287
AssociationFeeFrequency    231900
SubdivisionName            202363
MainLevelBedrooms          124728
Flooring                   110708
AssociationFee              93563
HighSchoolDistrict          80644
MLSAreaMajor                44008
Stories                     39369
Levels                      29245
GarageSpaces                11517
LotSizeAcres                 5353
LotSizeSquareFeet            5326
LotSizeArea                  5302
Latitude                     2712
Longitude                    2712
StreetNumberNumeric           362
ParkingTotal                  348
UnparsedAddress               294
City                          240
YearBuilt                     223
LivingArea                    164
BathroomsTotalInteger          50
PostalCode                      1
Name: Missing count, dtype: int64

In [20]:
# Drop school-name fields with limited coverage.
school_columns_to_drop = [
    "ElementarySchool",
    "MiddleOrJuniorSchool",
    "HighSchool",
]

property_data = property_data.drop(columns=school_columns_to_drop)

print("Remaining columns:", property_data.shape[1])
print("Remaining rows:", len(property_data))

Remaining columns: 33
Remaining rows: 309710


In [21]:
# Mark missing categorical values as Unknown.
categorical_columns = [
    "Flooring",
    "MLSAreaMajor",
    "SubdivisionName",
    "Levels",
    "HighSchoolDistrict",
]

for col in categorical_columns:
    property_data[col] = (
        property_data[col]
        .astype("string")
        .fillna("Unknown")
    )

# Verify that missing values have been handled.
print(property_data[categorical_columns].isna().sum())
print("Remaining rows:", len(property_data))

Flooring              0
MLSAreaMajor          0
SubdivisionName       0
Levels                0
HighSchoolDistrict    0
dtype: int64
Remaining rows: 309710


In [22]:
# Mark missing categorical values as Unknown.
categorical_columns = [
    "Flooring",
    "MLSAreaMajor",
    "SubdivisionName",
    "Levels",
    "HighSchoolDistrict",
]

for col in categorical_columns:
    property_data[col] = (
        property_data[col]
        .astype("string")
        .fillna("Unknown")
    )

# Verify that missing values have been handled.
print(property_data[categorical_columns].isna().sum())
print("Remaining rows:", len(property_data))

Flooring              0
MLSAreaMajor          0
SubdivisionName       0
Levels                0
HighSchoolDistrict    0
dtype: int64
Remaining rows: 309710


In [23]:
# Inspect HOA payment-frequency categories.
print(property_data["AssociationFeeFrequency"].value_counts(dropna=False))

# Check whether HOA fee and frequency are missing together.
display(
    pd.crosstab(
        property_data["AssociationFee"].isna(),
        property_data["AssociationFeeFrequency"].isna(),
        rownames=["Fee is missing"],
        colnames=["Frequency is missing"],
    )
)

AssociationFeeFrequency
NaN             231900
Monthly          68213
Annually          6306
Quarterly         2883
SemiAnnually       408
Name: count, dtype: int64


Frequency is missing,False,True
Fee is missing,,
False,77757,138390
True,53,93510


In [24]:
# Compare missing HOA fees and payment frequencies.
# True：missing；False：have values recorded
display(
    pd.crosstab(
        property_data["AssociationFee"].isna(),
        property_data["AssociationFeeFrequency"].isna(),
        rownames=["Fee is missing"],
        colnames=["Frequency is missing"],
    )
)

Frequency is missing,False,True
Fee is missing,,
False,77757,138390
True,53,93510


In [25]:
# Mark missing payment frequencies as Unknown.
property_data["AssociationFeeFrequency"] = (
    property_data["AssociationFeeFrequency"]
    .astype("string")
    .fillna("Unknown")
)

print(
    property_data["AssociationFeeFrequency"]
    .value_counts(dropna=False)
)

AssociationFeeFrequency
Unknown         231900
Monthly          68213
Annually          6306
Quarterly         2883
SemiAnnually       408
Name: count, dtype: Int64


In [26]:
# Show remaining missing values.
remaining_missing = property_data.isna().sum()

display(
    remaining_missing[remaining_missing > 0]
    .sort_values(ascending=False)
    .rename("Missing count")
)

MainLevelBedrooms        124728
AssociationFee            93563
Stories                   39369
GarageSpaces              11517
LotSizeAcres               5353
LotSizeSquareFeet          5326
LotSizeArea                5302
Longitude                  2712
Latitude                   2712
StreetNumberNumeric         362
ParkingTotal                348
UnparsedAddress             294
City                        240
YearBuilt                   223
LivingArea                  164
BathroomsTotalInteger        50
PostalCode                    1
Name: Missing count, dtype: int64

In [27]:
# Mark missing location categories as Unknown.
for col in ["City", "PostalCode"]:
    property_data[col] = (
        property_data[col]
        .astype("string")
        .fillna("Unknown")
    )

print(property_data[["City", "PostalCode"]].isna().sum())

City          0
PostalCode    0
dtype: int64


In [28]:
# Inspect numerical feature ranges and missing values.
#if garage space = 0: maybe no garage
numeric_columns = property_data.select_dtypes(
    include="number"
).columns

numeric_summary = property_data[numeric_columns].describe().T

numeric_summary["Missing count"] = (
    property_data[numeric_columns].isna().sum()
)

display(
    numeric_summary[
        ["Missing count", "min", "25%", "50%", "75%", "max"]
    ]
)

,Missing count,min,25%,50%,75%,max
Latitude,2712,-117.472493,33.766300,34.090283,35.110898,5.613037e+01
Longitude,2712,-177.646696,-119.299552,-118.040098,-117.265427,3.290000e+02
LivingArea,164,0.000000,1376.000000,1804.000000,2421.000000,1.237640e+05
ParkingTotal,348,-143.000000,2.000000,2.000000,3.000000,1.572000e+04
LotSizeAcres,5353,0.000000,0.130000,0.166300,0.237600,7.810698e+06
YearBuilt,223,1776.000000,1956.000000,1976.000000,1998.000000,2.026000e+03
StreetNumberNumeric,362,0.000000,1125.000000,3708.000000,13176.000000,7.814782e+07
BathroomsTotalInteger,50,0.000000,2.000000,2.000000,3.000000,1.750000e+02
BedroomsTotal,0,0.000000,3.000000,3.000000,4.000000,4.500000e+01
Stories,39369,1.000000,1.000000,1.000000,2.000000,2.000000e+00


In [29]:
# Right Now, Flag clearly invalid numerical values as missing.

In [30]:

invalid_rules = {
    "Latitude": ~property_data["Latitude"].between(-90, 90),
    "Longitude": ~property_data["Longitude"].between(-180, 180),
    "LivingArea": property_data["LivingArea"].le(0),
    "ParkingTotal": property_data["ParkingTotal"].lt(0),
}

for col, invalid in invalid_rules.items():
    invalid = invalid & property_data[col].notna()
    print(f"{col}: invalid values changed to NaN = {invalid.sum()}")
    property_data.loc[invalid, col] = np.nan

print("Remaining rows:", len(property_data))

Latitude: invalid values changed to NaN = 1
Longitude: invalid values changed to NaN = 1
LivingArea: invalid values changed to NaN = 126
ParkingTotal: invalid values changed to NaN = 74
Remaining rows: 309710


In [31]:
# Inspect suspicious construction years.
display(
    property_data.loc[
        property_data["YearBuilt"].gt(2026),
        ["UnparsedAddress", "YearBuilt", "NewConstructionYN"]
    ].head(20)
)

# Inspect main-level bedrooms exceeding total bedrooms.
display(
    property_data.loc[
        property_data["MainLevelBedrooms"].gt(
            property_data["BedroomsTotal"]
        ),
        ["UnparsedAddress", "MainLevelBedrooms", "BedroomsTotal"]
    ].head(20)
)

,UnparsedAddress,YearBuilt,NewConstructionYN


,UnparsedAddress,MainLevelBedrooms,BedroomsTotal
936,14242 Uxbridge Street,9.0,4.0
986,6074 Apache Drive,3.0,2.0
1856,849 W Casmalia Street,34.0,4.0
2035,9306 Degranvelle Drive,5.0,4.0
3051,2771 Rhoades Road,5.0,4.0
3401,9620 Cortez Avenue,4.0,3.0
3570,402 W 10th Street,4.0,3.0
3990,729 W 4th Street,2.0,1.0
5511,1841 E Avenue R5,4.0,3.0
5573,4522 Jutland Drive,4.0,3.0


In [32]:
# Count records with inconsistent bedroom counts.
bedroom_conflict = (
    property_data["MainLevelBedrooms"].notna()
    & property_data["BedroomsTotal"].notna()
    & property_data["MainLevelBedrooms"].gt(
        property_data["BedroomsTotal"]
    )
)

print("Rows with inconsistent bedroom counts:", bedroom_conflict.sum())
print(
    "Percentage of all rows:",
    round(bedroom_conflict.mean() * 100, 2)
)

Rows with inconsistent bedroom counts: 1870
Percentage of all rows: 0.6


In [33]:
# Mark conflicting main-level bedroom counts as missing.
property_data.loc[
    bedroom_conflict, "MainLevelBedrooms"
] = np.nan

print(
    "Missing MainLevelBedrooms:",
    property_data["MainLevelBedrooms"].isna().sum()
)
print("Remaining rows:", len(property_data))

Missing MainLevelBedrooms: 126598
Remaining rows: 309710


In [34]:
# Inspect the largest garage-space counts.
display(
    property_data.nlargest(10, "GarageSpaces")[
        ["UnparsedAddress", "GarageSpaces", "ParkingTotal", "LivingArea"]
    ]
)

# Inspect the largest total-parking counts.
display(
    property_data.nlargest(10, "ParkingTotal")[
        ["UnparsedAddress", "GarageSpaces", "ParkingTotal", "LivingArea"]
    ]
)

,UnparsedAddress,GarageSpaces,ParkingTotal,LivingArea
120115,7148 Rockford Court,600.0,600.0,2541.0
121362,2861 Pacific Ave,600.0,600.0,1138.0
172733,668 E Tangerine Street,500.0,500.0,4351.0
193595,13523 Algonquin Road,500.0,506.0,1756.0
387044,37885 High Ridge Drive,500.0,500.0,3290.0
45883,1509 Pole Line Road,400.0,400.0,925.0
94539,1999 Highland Drive,400.0,400.0,892.0
99378,6001 Brookmont Drive,400.0,403.0,1666.0
121057,131 Loftus,400.0,400.0,840.0
183584,824 S Sylvan Street,400.0,400.0,1145.0


,UnparsedAddress,GarageSpaces,ParkingTotal,LivingArea
360917,4138 Joan Ave,NaN,15720.0,1160.0
354333,2992 Matilija canyon road Road,NaN,10009.0,1050.0
8952,16428 Avalon Avenue,NaN,9775.0,1254.0
72960,1305 S Thomas Street,NaN,6154.0,837.0
598212,135 W 9th Ave,NaN,3921.0,938.0
324679,21952 Avenida De Arboles,NaN,2065.0,8316.0
228437,9971 Foster,NaN,1949.0,1262.0
228326,29502 Catano Road,2.0,1419.0,1417.0
395380,2817 Fairfield Avenue,NaN,1111.0,876.0
184999,5476 Cook Lane,6.0,1006.0,1032.0


In [35]:
# Inspect unusually large garage and parking counts.
for col in ["GarageSpaces", "ParkingTotal"]:
    large_values = property_data.loc[
        property_data[col].gt(20), col
    ]

In [36]:
# Treat parking counts above 100 as unreliable.
for col in ["GarageSpaces", "ParkingTotal"]:
    invalid = property_data[col].gt(100)

    print(f"{col}: values changed to NaN = {invalid.sum()}")
    property_data.loc[invalid, col] = np.nan

print("Remaining rows:", len(property_data))

GarageSpaces: values changed to NaN = 24
ParkingTotal: values changed to NaN = 126
Remaining rows: 309710


In [37]:
# Convert sale dates and inspect their coverage.
sale_dates = pd.to_datetime(sale_dates, errors="coerce")

print("Missing or invalid sale dates:", sale_dates.isna().sum())
print("Earliest sale date:", sale_dates.min())
print("Latest sale date:", sale_dates.max())

Missing or invalid sale dates: 0
Earliest sale date: 2024-01-01 00:00:00
Latest sale date: 2026-04-30 00:00:00


In [38]:

# split data, use the most recent month of available data as the test set, 

In [39]:
# Split by sale date.
test_start = pd.Timestamp("2026-04-01")

train_mask = sale_dates.lt(test_start)
test_mask = sale_dates.ge(test_start)

X_train = property_data.loc[train_mask].copy()
X_test = property_data.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()


In [40]:
# List all retained columns.
print("Number of retained columns:", property_data.shape[1])

for number, col in enumerate(property_data.columns, start=1):
    print(f"{number}. {col}")

Number of retained columns: 33
1. Flooring
2. ViewYN
3. PoolPrivateYN
4. Latitude
5. Longitude
6. UnparsedAddress
7. PropertyType
8. LivingArea
9. AssociationFeeFrequency
10. MLSAreaMajor
11. CountyOrParish
12. AttachedGarageYN
13. ParkingTotal
14. PropertySubType
15. LotSizeAcres
16. SubdivisionName
17. YearBuilt
18. StreetNumberNumeric
19. BathroomsTotalInteger
20. City
21. BedroomsTotal
22. StateOrProvince
23. FireplaceYN
24. Stories
25. Levels
26. LotSizeArea
27. MainLevelBedrooms
28. NewConstructionYN
29. GarageSpaces
30. HighSchoolDistrict
31. PostalCode
32. AssociationFee
33. LotSizeSquareFeet


In [41]:
#impute missing values

# First batch of numeric columns to impute.
numeric_columns_to_impute = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "MainLevelBedrooms",
    "GarageSpaces",
    "ParkingTotal",
]

# Calculate medians using training data only.
training_medians = X_train[numeric_columns_to_impute].median()

# Fill missing values using the same training medians.
X_train[numeric_columns_to_impute] = (
    X_train[numeric_columns_to_impute].fillna(training_medians)
)

X_test[numeric_columns_to_impute] = (
    X_test[numeric_columns_to_impute].fillna(training_medians)
)

# Show the values used for filling.
print("Training medians:")
print(training_medians)

# Verify remaining missing values.
print("\nRemaining missing values — training:")
print(X_train[numeric_columns_to_impute].isna().sum())

print("\nRemaining missing values — test:")
print(X_test[numeric_columns_to_impute].isna().sum())

Training medians:
LivingArea               1803.0
BedroomsTotal               3.0
BathroomsTotalInteger       2.0
YearBuilt                1976.0
Stories                     1.0
MainLevelBedrooms           3.0
GarageSpaces                2.0
ParkingTotal                2.0
dtype: float64

Remaining missing values — training:
LivingArea               0
BedroomsTotal            0
BathroomsTotalInteger    0
YearBuilt                0
Stories                  0
MainLevelBedrooms        0
GarageSpaces             0
ParkingTotal             0
dtype: int64

Remaining missing values — test:
LivingArea               0
BedroomsTotal            0
BathroomsTotalInteger    0
YearBuilt                0
Stories                  0
MainLevelBedrooms        0
GarageSpaces             0
ParkingTotal             0
dtype: int64


In [42]:
# Check remaining missing values in both datasets.
remaining_missing = pd.DataFrame({
    "Training missing": X_train.isna().sum(),
    "Test missing": X_test.isna().sum(),
})

display(
    remaining_missing.loc[
        remaining_missing.gt(0).any(axis=1)
    ].sort_values("Training missing", ascending=False)
)

,Training missing,Test missing
AssociationFee,89845,3718
LotSizeAcres,5110,243
LotSizeSquareFeet,5084,242
LotSizeArea,5060,242
Latitude,2633,80
Longitude,2633,80
StreetNumberNumeric,336,26
UnparsedAddress,271,23


In [43]:
# Fill missing lot square footage using recorded acreage.
for name, dataset in [("Training", X_train), ("Test", X_test)]:
    can_convert = (
        dataset["LotSizeSquareFeet"].isna()
        & dataset["LotSizeAcres"].gt(0)
        & np.isfinite(dataset["LotSizeAcres"])
    )

    dataset.loc[can_convert, "LotSizeSquareFeet"] = (
        dataset.loc[can_convert, "LotSizeAcres"] * 43560
    )

    print(f"{name}: values filled from acreage = {can_convert.sum()}")
    print(
        f"{name}: remaining missing LotSizeSquareFeet = "
        f"{dataset['LotSizeSquareFeet'].isna().sum()}"
    )

# Retain one lot-area feature.
X_train = X_train.drop(columns=["LotSizeAcres", "LotSizeArea"])
X_test = X_test.drop(columns=["LotSizeAcres", "LotSizeArea"])

print("\nTraining columns:", X_train.shape[1])
print("Test columns:", X_test.shape[1])

Training: values filled from acreage = 20
Training: remaining missing LotSizeSquareFeet = 5064
Test: values filled from acreage = 0
Test: remaining missing LotSizeSquareFeet = 242

Training columns: 31
Test columns: 31


In [44]:
# Calculate the lot-area median from training data only.
lot_median = X_train["LotSizeSquareFeet"].median()

# Fill remaining missing values in both datasets.
X_train["LotSizeSquareFeet"] = (
    X_train["LotSizeSquareFeet"].fillna(lot_median)
)
X_test["LotSizeSquareFeet"] = (
    X_test["LotSizeSquareFeet"].fillna(lot_median)
)

print("Training lot-area median:", lot_median)
print("Training missing:", X_train["LotSizeSquareFeet"].isna().sum())
print("Test missing:", X_test["LotSizeSquareFeet"].isna().sum())

Training lot-area median: 7245.0
Training missing: 0
Test missing: 0


In [45]:
# Remove address fields from model inputs.
address_columns = ["UnparsedAddress", "StreetNumberNumeric"]

X_train = X_train.drop(columns=address_columns)
X_test = X_test.drop(columns=address_columns)

print("Training columns:", X_train.shape[1])
print("Test columns:", X_test.shape[1])

Training columns: 29
Test columns: 29


In [46]:
# Calculate coordinate medians by postal code from training data.
coordinate_columns = ["Latitude", "Longitude"]

postal_coordinate_medians = (
    X_train.loc[X_train["PostalCode"].ne("Unknown")]
    .groupby("PostalCode")[coordinate_columns]
    .median()
)

# Fill missing coordinates using the same training lookup.
for dataset in [X_train, X_test]:
    for col in coordinate_columns:
        estimated_coordinates = dataset["PostalCode"].map(
            postal_coordinate_medians[col]
        )

        dataset[col] = dataset[col].fillna(estimated_coordinates)

# Check remaining missing coordinates.
print("Training missing coordinates:")
print(X_train[coordinate_columns].isna().sum())

print("\nTest missing coordinates:")
print(X_test[coordinate_columns].isna().sum())

Training missing coordinates:
Latitude     60
Longitude    60
dtype: int64

Test missing coordinates:
Latitude     1
Longitude    1
dtype: int64


In [47]:
# Keep rows with both coordinates available.
train_valid = X_train[["Latitude", "Longitude"]].notna().all(axis=1)
test_valid = X_test[["Latitude", "Longitude"]].notna().all(axis=1)

# Show how many rows will be removed.
print("Training rows removed:", (~train_valid).sum())
print("Test rows removed:", (~test_valid).sum())

# Keep the same rows in features and targets.
X_train = X_train.loc[train_valid]
y_train = y_train.loc[train_valid]

X_test = X_test.loc[test_valid]
y_test = y_test.loc[test_valid]

print("Remaining training rows:", len(X_train))
print("Remaining test rows:", len(X_test))

Training rows removed: 60
Test rows removed: 1
Remaining training rows: 297161
Remaining test rows: 12488


In [48]:
# Summarize HOA fees by payment frequency in training data.
display(
    X_train.groupby("AssociationFeeFrequency")["AssociationFee"]
    .agg(["size", "count", "median"])
    .rename(columns={
        "size": "Total rows",
        "count": "Rows with fee",
        "median": "Median fee",
    })
)

,Total rows,Rows with fee,Median fee
AssociationFeeFrequency,,,
Annually,6041,6014,393.5
Monthly,65312,65290,178.0
Quarterly,2741,2740,260.0
SemiAnnually,396,396,179.0
Unknown,222671,132921,0.0


In [49]:
# Drop HOA fields because 74.9% of training records have an unknown payment frequency,
# preventing reliable conversion of fees to a consistent monthly amount. 

In [50]:
# Drop HOA fields if they are still present.
hoa_columns = ["AssociationFee", "AssociationFeeFrequency"]

X_train = X_train.drop(columns=hoa_columns, errors="ignore")
X_test = X_test.drop(columns=hoa_columns, errors="ignore")

print("Training columns:", X_train.shape[1])
print("Test columns:", X_test.shape[1])

Training columns: 27
Test columns: 27


In [51]:
# Convert categorical fields to numeric

In [52]:
# Identify categorical columns.
categorical_columns = X_train.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

# Count distinct values in each categorical column.
display(
    X_train[categorical_columns]
    .nunique()
    .sort_values(ascending=False)
    .rename("Number of categories")
)

SubdivisionName       11664
PostalCode             3035
City                   1096
MLSAreaMajor           1070
HighSchoolDistrict      442
Flooring                316
CountyOrParish           63
Levels                   19
ViewYN                    3
PoolPrivateYN             3
AttachedGarageYN          3
FireplaceYN               3
NewConstructionYN         3
PropertyType              1
PropertySubType           1
StateOrProvince           1
Name: Number of categories, dtype: int64

In [53]:
# Inspect state values in both datasets.
print("Training state values:")
print(X_train["StateOrProvince"].value_counts(dropna=False))

print("\nTest state values:")
print(X_test["StateOrProvince"].value_counts(dropna=False))

Training state values:
StateOrProvince
CA    297161
Name: count, dtype: int64

Test state values:
StateOrProvince
CA    12488
Name: count, dtype: int64


In [58]:
# Remove constant scope fields from model inputs.
scope_columns = [
    "StateOrProvince",
    "PropertyType",
    "PropertySubType",
]

X_train = X_train.drop(columns=scope_columns, errors="ignore")
X_test = X_test.drop(columns=scope_columns, errors="ignore")

print("Training columns:", X_train.shape[1])
print("Test columns:", X_test.shape[1])

Training columns: 24
Test columns: 24


In [59]:
# Identify remaining categorical columns.
categorical_columns = X_train.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

# Standardize category formatting in both datasets.
for dataset in [X_train, X_test]:
    for col in categorical_columns:
        dataset[col] = (
            dataset[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace("", pd.NA)
            .fillna("unknown")
        )

# Review category counts after standardization.
display(
    X_train[categorical_columns]
    .nunique()
    .sort_values(ascending=False)
    .rename("Number of categories")
)

SubdivisionName       11248
PostalCode             3035
City                   1091
MLSAreaMajor           1070
HighSchoolDistrict      442
Flooring                316
CountyOrParish           63
Levels                   19
ViewYN                    3
PoolPrivateYN             3
AttachedGarageYN          3
FireplaceYN               3
NewConstructionYN         3
Name: Number of categories, dtype: int64

In [60]:
%pip install scikit-learn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Identify numeric columns.
numeric_columns = [
    col for col in X_train.columns
    if col not in categorical_columns
]

# Configure categorical encoding and retain numeric features.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            categorical_columns,
        ),
        ("numeric", "passthrough", numeric_columns),
    ],
    sparse_threshold=1.0,
)

# Learn categories from training data only.
X_train_encoded = preprocessor.fit_transform(X_train)

# Apply the same encoding to test data.
X_test_encoded = preprocessor.transform(X_test)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded test shape:", X_test_encoded.shape)

Note: you may need to restart the kernel to use updated packages.
Encoded training shape: (297161, 17310)
Encoded test shape: (12488, 17310)


In [61]:
from pathlib import Path

# Use the existing project folder.
OUTPUT_DIR = (
    Path.home()
    / "Downloads"
    / "IDX-Exchange-Data-Science"
)

# Check the folder and data alignment.
assert OUTPUT_DIR.is_dir(), f"Project folder not found: {OUTPUT_DIR}"
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert X_train.columns.equals(X_test.columns)

# Include the target and sale date for export only.
train_export = X_train.assign(
    ClosePrice=y_train,
    CloseDate=sale_dates.loc[X_train.index],
)

test_export = X_test.assign(
    ClosePrice=y_test,
    CloseDate=sale_dates.loc[X_test.index],
)

# Save cleaned training and test CSV files.
train_path = OUTPUT_DIR / "single_family_train_cleaned.csv"
test_path = OUTPUT_DIR / "single_family_test_cleaned.csv"

train_export.to_csv(train_path, index=False)
test_export.to_csv(test_path, index=False)

print("Training CSV saved:", train_path)
print("Training shape:", train_export.shape)

print("\nTest CSV saved:", test_path)
print("Test shape:", test_export.shape)

Training CSV saved: /Users/jasmineyuan/Downloads/IDX-Exchange-Data-Science/single_family_train_cleaned.csv
Training shape: (297161, 26)

Test CSV saved: /Users/jasmineyuan/Downloads/IDX-Exchange-Data-Science/single_family_test_cleaned.csv
Test shape: (12488, 26)


In [62]:
import joblib

# Verify that encoded features match the target lengths.
assert X_train_encoded.shape[0] == len(y_train)
assert X_test_encoded.shape[0] == len(y_test)

# Save encoded features, targets, and encoding rules together.
modeling_data = {
    "X_train": X_train_encoded,
    "X_test": X_test_encoded,
    "y_train": y_train.to_numpy(),
    "y_test": y_test.to_numpy(),
    "preprocessor": preprocessor,
}

joblib.dump(
    modeling_data,
    OUTPUT_DIR / "week3_modeling_data.joblib",
    compress=3,
)

print("Modeling data saved.")

Modeling data saved.
